# LOAD

In [ ]:
import os
import random
import cv2 as cv 
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np  # Biblioteca NumPy para manipulação de arrays numéricos (imagens são arrays)
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
from imblearn.over_sampling import SMOTE
from collections import Counter

# modelos de testes
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import ExtraTreesClassifier
import importlib, utils
importlib.reload(utils)     # recarrega o módulo
from utils import Utils     # reimporta a classe (rebind!)

In [ ]:
# Lê algumas imagens para teste cv.imread()
caminho_imagens = Utils.carregarCaminhoImagens(r'../../data/originais')
numero_imagens_amostra = 5
total = len(caminho_imagens)
imagens_carregadas = []

# all_images = [cv.imread(caminho) for caminho in caminho_imagens]            

for i in range(numero_imagens_amostra):
    # seleciona aleatoriamente um número entre 0 e a quantidade total de imagens.    
    x = random.randint(0, len(caminho_imagens))

    img = cv.imread(caminho_imagens[x])
    img_resized = cv.resize(img, (1280, 720))

    # Carrega a imagem usando o opencv e adiciona em forma de tupla a imagem na posição 0 e o nome da imagem na posição 1 (o nome é usado para legenda depois.)
    imagens_carregadas.append((img_resized, os.path.basename(caminho_imagens[x])))

In [ ]:
# --- REMOVE BOARDAS E ADICIONA EM OUTRA LISTA PARA ANALISE ---

lista_imagem_poligono = []
lista_imagem_poligono_cinza = []

for arquivo in caminho_imagens:
    imagem = cv.imread(arquivo)
    imagem_gray = cv.imread(arquivo, 0)

    imagem = cv.resize(imagem, (1280, 720))
    imagem_gray = cv.resize(imagem_gray, (1280, 720))

    #    Isso zera tudo que está fora do polígono.
    imagem_apenas_box = Utils.aplicarPoligono(imagem)
    imagem_apenas_box_cinza = Utils.aplicarPoligono(imagem_gray)
    
    lista_imagem_poligono.append((imagem_apenas_box, arquivo))
    lista_imagem_poligono_cinza.append((imagem_apenas_box_cinza, arquivo))

len(lista_imagem_poligono_cinza)

# PLOTS HISTOGRAMAS

## GREY SCALE

In [ ]:
# GrayScale

if 'imagens_carregadas' in globals() and len(imagens_carregadas) > 0:
    imagem_cinza = []
    nomes = []
    for idx, (imagem_original_bgr, nome_imagem) in enumerate(imagens_carregadas):
        # Converte a imagem BGR para tons de cinza.
        imagem_cinza.append(cv.cvtColor(imagem_original_bgr, cv.COLOR_BGR2GRAY))
        nomes.append(nome_imagem)
        
    Utils.plotarColuna(imagem_cinza, nomes)        
else:
    print("Nenhuma imagem carregada para processar em tons de cinza. Execute a célula de carregamento de amostra primeiro.")

## HISTOGRAMA E O ADAPTIVE THRESHOLD

In [ ]:
### histograma e o adaptiveThreshold
imagens = []
nomes = []
for i in range(numero_imagens_amostra):
    idx = random.randint(0, len(lista_imagem_poligono_cinza) -1)
    img = lista_imagem_poligono_cinza[idx][0]
    imagens.append(img)    
    img_h, _ = Utils.gerarHistograma(img, lista_imagem_poligono_cinza[idx][1], Utils.mascaraPoligono(img))
    imagens.append(img_h)
    imagens.append(cv.adaptiveThreshold(img, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 11, 2))    
    nomes.append(lista_imagem_poligono_cinza[i][1])
    
Utils.plotarColuna(imagens, nomes, 3)

## RGB e HSV

In [ ]:
imagens = []

# RGB e HSV
for idx in range(numero_imagens_amostra):    
    # Converte para RGB para visualização correta com matplotlib
    # imagem_rgb = cv.cvtColor(imagens_carregadas[idx][0], cv.COLOR_BGR2RGB)
    idx = random.randint(0, len(lista_imagem_poligono)-1)

    imagem_rgb = lista_imagem_poligono[idx][0]
    imagens.append(imagem_rgb)
    vetor_histograma_rgb = Utils.gerarHistogramaRgb(imagem_rgb, Utils.mascaraPoligono(imagem_rgb))
    hist_b, hist_g, hist_r = np.split(vetor_histograma_rgb, 3)
    imagens.append(Utils.histogramaRgbParaImagem(hist_b, hist_g, hist_r))


    # Converte para HSV
    imagem_hsv = cv.cvtColor(lista_imagem_poligono[idx][0], cv.COLOR_BGR2HSV)
    imagens.append(imagem_hsv)
    vetor_histograma_hsv = Utils.gerarHistogramaHsv(imagem_hsv, Utils.mascaraPoligono(imagem_hsv))
    hist_h, hist_s, hist_v = np.split(vetor_histograma_hsv, 3)

    # hist_h = cv.calcHist([imagem_hsv], [0], Utils.mascaraPoligono(imagem_hsv), [256], [0,256]).ravel()
    # hist_s = cv.calcHist([imagem_hsv], [1], Utils.mascaraPoligono(imagem_hsv), [256], [0,256]).ravel()
    # hist_v = cv.calcHist([imagem_hsv], [2], Utils.mascaraPoligono(imagem_hsv), [256], [0,256]).ravel()
    imagens.append(Utils.histogramaHsvParaImagem(hist_h, hist_s, hist_v))

Utils.plotarColuna(imagens, None, 4)

# DATASETS

### CARREGAR CAMINHOS

In [ ]:
histograma_rgb = r'..\datasets\histograma_rgb_original.npz'
histograma_rgb_poligono = r'..\datasets\histograma_rgb_poligono.npz'
histograma_hsv = r'..\datasets\histograma_hsv_original.npz'
histograma_hsv_poligono = r'..\datasets\histograma_hsv_poligono.npz'
histograma_gray = r'..\datasets\histograma_gray_original.npz'
histograma_gray_poligono = r'..\datasets\histograma_gray_poligono.npz'

### RGB HSV GRAY

In [ ]:
# Dataset histograma imagem rgb com bordas
X_rgb, y_rgb = [], []
X_hsv, y_hsv = [], []
X_gray, y_gray = [],[]

for arquivo in caminho_imagens:
    Utils.tratarImagemRgb(arquivo, X_rgb, y_rgb)
    Utils.tratarImagemHsv(arquivo, X_hsv, y_hsv)
    Utils.tratarImagemGray(arquivo, X_gray, y_gray)

# Converte para arrays numpy
X_rgb = np.array(X_rgb)
y_rgb = np.array(y_rgb)
X_hsv = np.array(X_hsv)
y_hsv = np.array(y_hsv)
X_gray = np.array(X_gray)
y_gray = np.array(y_gray)


# Dataset histograma imagem rgb sem bordas
X_rgb_poligono, y_rgb_poligono = [],[]
X_hsv_poligono, y_hsv_poligono = [],[]
X_gray_poligono, y_gray_poligono = [],[]

for imagem_bgr, classe in lista_imagem_poligono:  # mesma estrutura usada antes    
    Utils.tratarImagemRgbPoligono(classe, X_rgb_poligono, y_rgb_poligono, imagem_bgr)
    Utils.tratarImagemHsvPoligono(classe, X_hsv_poligono, y_hsv_poligono, imagem_bgr)
    Utils.tratarImagemGrayPoligono(classe, X_gray_poligono, y_gray_poligono, imagem_bgr)

# Converte para arrays numpy
X_rgb_poligono = np.array(X_rgb_poligono)
y_rgb_poligono = np.array(y_rgb_poligono)
X_hsv_poligono = np.array(X_hsv_poligono)
y_hsv_poligono = np.array(y_hsv_poligono)
X_gray_poligono = np.array(X_gray_poligono)
y_gray_poligono = np.array(y_gray_poligono)


# Salva em .npz
np.savez(histograma_rgb, X=X_rgb, y=y_rgb)
np.savez(histograma_rgb_poligono, X=X_rgb_poligono, y=y_rgb_poligono)
np.savez(histograma_hsv, X=X_hsv, y=y_hsv)
np.savez(histograma_hsv_poligono, X=X_hsv_poligono, y=y_hsv_poligono)
np.savez(histograma_gray, X=X_gray, y=y_gray)
np.savez(histograma_gray_poligono, X=X_gray_poligono, y=y_gray_poligono)

In [ ]:
dataset_files = {
    'RGB': histograma_rgb,
    'RGB_POLIGONO': histograma_rgb_poligono,
    'HSV': histograma_hsv,
    'HSV_POLIGONO': histograma_hsv_poligono,
    'GRAY': histograma_gray, 
    'GRAY_POLIGONO': histograma_gray_poligono,
}

funcoes_imagens = {
        'RGB': Utils.tratarImagemRgb,
        'RGB_POLIGONO': Utils.tratarImagemRgbPoligono,
        'HSV': Utils.tratarImagemHsv,
        'HSV_POLIGONO': Utils.tratarImagemHsvPoligono,
        'GRAY': Utils.tratarImagemGray,
        'GRAY_POLIGONO': Utils.tratarImagemGrayPoligono,
}

nome_dataset = ['RGB', 'RGB_POLIGONO', 'HSV', 'HSV_POLIGONO', 'GRAY', 'GRAY_POLIGONO']

# TESTE

In [ ]:
def testarModeloSplit(modelo):
    teste = 0.2
    ds = []

    for dataset in nome_dataset:
        X, y = Utils.carregaDataset(dataset_files[dataset])
        ds.append(Utils.testarModelosPipelineSplit(X, y, teste, modelo, dataset))
    return pd.concat(ds, ignore_index=True)

def testarModeloCV(modelo):
    ds = []

    for dataset in nome_dataset:
        X, y = Utils.carregaDataset(dataset_files[dataset])
        ds.append(Utils.testarModelosPipelineCV(modelo, X, y, dataset))
    return pd.concat(ds, ignore_index=True)

In [ ]:
# modelo = None
# df_all_split = testarModeloSplit(modelo)
# df_all_split.to_csv("../datasets/df_all_split.csv")

In [ ]:
df_all_split = pd.read_csv("../datasets/df_all_split.csv")

# df_all_split["score"] = (0.6*df_all_split["Recall"] + 
#                          0.2*df_all_split["Precision"] + 
#                          0.2*df_all_split["F1"]) / 3
df_top_split = df_all_split.sort_values("Recall", ascending=False).head(20)
# Utils.matrizConfusao(df_all_split)
df_top_split[['Modelo', 'Dataset', 'Pipeline', 'Recall']]

In [ ]:
len(df_all_split)

In [ ]:
df_plot = df_all_split[(df_all_split['Modelo'] == 'RandomForest') & ((df_all_split['Dataset'] == 'HSV') | (df_all_split['Dataset'] == 'RGB'))]
df_plot[['Modelo', 'Dataset', 'Pipeline', 'Recall']]

In [ ]:
# modelo = None
# df_all_cv = testarModeloCV(modelo)
# df_all_cv.to_csv("../datasets/df_all_cv.csv")

In [ ]:
df_all_cv = pd.read_csv("../datasets/df_all_cv.csv")

# df_all_cv["score"] = ( 0.6 * df_all_cv["CV_test_recall_macro_mean"] +
#                         0.2 * df_all_cv["CV_test_precision_macro_mean"] +
#                         0.2 * df_all_cv["CV_test_f1_macro_mean"] -
#                         0.2 * df_all_cv["CV_test_recall_macro_std"]) / 3

df_top_cv = df_all_cv.sort_values("CV_test_recall_macro_mean", ascending=False).head(20)
df_top_cv["Recall"] = df_top_cv["CV_test_recall_macro_mean"]
# Utils.matrizConfusao(df_all_split)
# df_all_cv[["Modelo", "Dataset", "Pipeline", "CV_test_recall_macro_mean", "CV_test_precision_macro_mean", "CV_test_f1_macro_mean", "CV_test_recall_macro_std"]]
df_top_cv

In [ ]:
len(df_all_cv)

In [ ]:
df_union = pd.concat([df_top_split, df_top_cv], ignore_index=True)
df_drop = df_union.sort_values('Recall', ascending=False).drop_duplicates(subset=["Modelo", "Dataset", "Pipeline"])
df_union = df_drop[ ["Modelo", "Dataset", "Pipeline", "Recall"] ].sort_values('Recall', ascending=False)
df_union

In [ ]:
import matplotlib.pyplot as plt

# Se quiser limitar (ex.: Top 10 melhores)
df_plot = df_union.head(10).copy()

# Cria label mais legível (modelo em cima, dataset|pipeline embaixo)
df_plot['Label'] = (
    df_plot['Modelo'] + "\n" +
    df_plot['Dataset'] + " | " + df_plot['Pipeline']
)

plt.figure(figsize=(14,8))
bars = plt.barh(df_plot['Label'], df_plot['Recall']*100, color="skyblue")

# Adiciona os valores dentro da barra (centralizados)
for bar in bars:
    width = bar.get_width()
    plt.text(width/2,                           # posição X → meio da barra
             bar.get_y() + bar.get_height()/2,  # posição Y → meio da barra
             f"{width:.1f}%", 
             ha="center", va="center", 
             color="white", fontsize=12, fontweight="bold")

# Ajustes do gráfico
plt.xlabel("Recall (%)", fontweight="bold", fontsize=12)
plt.ylabel("Modelo | Dataset | Pipeline", fontweight="bold", fontsize=12)
plt.title("Top 10 combinações selecionadas por Recall", fontweight="bold", fontsize=14)

plt.yticks(fontweight="bold", fontsize=11)
plt.gca().invert_yaxis()  # coloca o maior no topo
plt.show()




## TREINO COM OS MELHORES

In [ ]:
for i in range(len(df_union)):
    modelo, dataset, pipeline = df_union.iloc[i]['Modelo'], df_union.iloc[i]['Dataset'], df_union.iloc[i]['Pipeline']
    X_train, y_train = Utils.carregaDataset(dataset_files[dataset])
    Utils.treinarMelhoresModelos(X_train, y_train, modelo=modelo, pipeline=pipeline, dataset=dataset)


# CLASSIFICAÇÃO

In [ ]:
listaImagens = Utils.carregarCaminhoImagens(r'..\dados\classesTeste')
order = sorted(listaImagens)
modelo = None
df = []


dsResultado = []
for caminho in order:
    for i in range(len(df_union)):
        modelo, dataset, pipeline = df_union.iloc[i]['Modelo'], df_union.iloc[i]['Dataset'], df_union.iloc[i]['Pipeline']
        X_class, y_class = [], []
        funcoes_imagens[dataset](caminho, X_class, y_class)
        X_class = np.array(X_class)
        dsResultado.append(Utils.classificarPipeline(modelo, dataset, pipeline, X_class, caminho))

df = pd.concat(dsResultado)
print(len(df['Imagem'].unique()))


In [ ]:
df['Ocupacao'] = df['Imagem'].str.extract(r'(\d+(?:[.,]\d+)?)\s*%')[0]
df['Minimo_Classe'] = df['Classe'].str.split('to', expand=True).astype('Int64')[0]
df['Maximo_Classe'] = df['Classe'].str.split('to', expand=True).astype('Int64')[1]
df['Acertou'] = (df['Ocupacao'].astype('int64') >= df['Minimo_Classe'].astype('int64')) & (df['Ocupacao'].astype('int64') < df['Maximo_Classe'].astype('int64'))  # booleana


In [ ]:
df.groupby(['Modelo', 'Dataset', 'Pipeline'])['Acertou'].mean().mul(100).round(1).reset_index().sort_values('Acertou', ascending=False)

In [ ]:
plt.figure(figsize=(10,6))
bars = plt.barh(df_top10['Label'], df_top10['Acertou'], color="skyblue")

# Adiciona os valores dentro das barras
for bar in bars:
    width = bar.get_width()
    plt.text(width - 5, 
             bar.get_y() + bar.get_height()/2, 
             f"{width:.1f}%", 
             ha="right", va="center", color="white", fontsize=10, fontweight="bold")

plt.xlabel("Acerto (%)", fontweight="bold")
plt.ylabel("Modelo | Dataset | Pipeline", fontweight="bold")
plt.title("Top 10 combinações por acerto (%)", fontweight="bold")

plt.yticks(fontweight="bold")  # <<< deixa os labels do eixo Y em negrito
plt.gca().invert_yaxis()
plt.show()


In [ ]:
df.groupby(['Modelo', 'Dataset', 'Pipeline', 'Classe'])['Acertou'].mean().mul(100).round(1).reset_index().sort_values('Acertou', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# Agrupa e calcula médias
df_grouped = (
    df.groupby(['Modelo', 'Dataset', 'Pipeline', 'Classe'])['Acertou']
      .mean().mul(100).round(1)
      .reset_index()
      .sort_values('Acertou', ascending=False)
)

# Pega só o Top 10 para visualização
df_top10 = df_grouped.head(10)

# Cria uma label única para exibir no eixo Y
df_top10['Label'] = (
    df_top10['Modelo'] + " | " +
    df_top10['Dataset'] + " | " +
    df_top10['Pipeline'] + " | Classe " +
    df_top10['Classe'].astype(str)
)

# Plot
plt.figure(figsize=(12,6))
bars = plt.barh(df_top10['Label'], df_top10['Acertou'], color="steelblue")

# Adiciona valores dentro das barras
for bar in bars:
    width = bar.get_width()
    plt.text(width - 5,
             bar.get_y() + bar.get_height()/2,
             f"{width:.1f}%",
             ha="right", va="center", color="white",
             fontsize=10, fontweight="bold")

# Ajustes do gráfico
plt.xlabel("Acerto (%)", fontweight="bold")
plt.ylabel("Modelo | Dataset | Pipeline | Classe", fontweight="bold")
plt.title("Top 10 combinações por classe (acerto %)", fontweight="bold")

plt.yticks(fontweight="bold")   # deixa os rótulos do eixo Y em negrito
plt.gca().invert_yaxis()        # melhor no estilo ranking
plt.show()
